# Business Operations Agent with HiveAgent MCP

This notebook shows how to build a business operations agent that connects Claude to [HiveAgent's MCP server](https://hiveagentiq.com) — a production endpoint with 743 tools across 36 industry verticals, from insurance claims to construction permits to international trade compliance.

**What this demonstrates:**
1. **Discovery** — find tools by natural language query
2. **Agent wallets** — create autonomous payment accounts
3. **Payment processing** — pay any entity on any rail
4. **Smart routing** — compare protocols and route optimally

No API keys are required for HiveAgent — the MCP endpoint is open.

## Setup

This notebook uses only two dependencies:

- **`anthropic`** — the Claude SDK for building the agent
- **`httpx`** — a standard async HTTP client for making raw JSON-RPC calls to the MCP server

No third-party MCP SDKs are needed. We speak directly to HiveAgent's MCP endpoint over HTTP.

In [ ]:
%pip install -q anthropic httpx

In [ ]:
import anthropic
import httpx
import json
import asyncio

# HiveAgent MCP endpoint — 743 tools, 36 verticals, no API key required
HIVEAGENT_MCP_URL = "https://hiveagentiq.com/mcp"

## Connect to HiveAgent MCP — Raw JSON-RPC over HTTP

The [Model Context Protocol (MCP)](https://modelcontextprotocol.io) uses JSON-RPC 2.0 over HTTP. Every tool call is a `POST` request with a structured payload:

```json
{
  "jsonrpc": "2.0",
  "id": 1,
  "method": "tools/call",
  "params": {
    "name": "tool_name",
    "arguments": { ... }
  }
}
```

We implement this directly with `httpx` — no MCP SDK required.

In [ ]:
async def call_mcp_tool(tool_name: str, arguments: dict = None) -> dict:
    """Make a JSON-RPC 2.0 call to the HiveAgent MCP server via raw httpx POST."""
    async with httpx.AsyncClient(timeout=30) as client:
        response = await client.post(
            HIVEAGENT_MCP_URL,
            json={
                "jsonrpc": "2.0",
                "id": 1,
                "method": "tools/call",
                "params": {
                    "name": tool_name,
                    "arguments": arguments or {}
                }
            },
            headers={"Content-Type": "application/json"}
        )
        response.raise_for_status()
        result = response.json()

    if "error" in result:
        raise RuntimeError(f"MCP error [{result['error'].get('code')}]: {result['error'].get('message')}")

    # MCP wraps tool output in a content envelope; parse the inner text if present
    payload = result.get("result", {})
    if "content" in payload and payload["content"]:
        text = payload["content"][0].get("text", "{}")
        if text.startswith("{") or text.startswith("["):
            return json.loads(text)
        return {"text": text}
    return payload


async def list_mcp_tools() -> list:
    """Retrieve the full list of tools from the MCP server."""
    async with httpx.AsyncClient(timeout=30) as client:
        response = await client.post(
            HIVEAGENT_MCP_URL,
            json={"jsonrpc": "2.0", "id": 1, "method": "tools/list", "params": {}},
            headers={"Content-Type": "application/json"}
        )
        response.raise_for_status()
    return response.json().get("result", {}).get("tools", [])


print("Helper functions defined.")

## Discover Available Tools

HiveAgent exposes a `hiveagent_discover` meta-tool that lets you search for relevant tools using natural language. This is how an agent would explore the 743-tool catalog without needing to enumerate every option.

In [ ]:
# First, verify connectivity and tool count
tools = await list_mcp_tools()
print(f"Connected to HiveAgent MCP — {len(tools)} tools available")

# Show a sample of tool categories
sample_tools = tools[:10]
print("\nSample tools:")
for t in sample_tools:
    name = t.get("name", "")
    desc = t.get("description", "")[:70]
    print(f"  {name:<40} {desc}")

In [ ]:
# Discover tools for payment operations using natural language
discovery_result = await call_mcp_tool(
    "hiveagent_discover",
    {"query": "process payments and manage agent wallets"}
)

matches = discovery_result.get("matches", [])
print(f"Found {len(matches)} matching tools for 'process payments and manage agent wallets':\n")
for match in matches[:8]:
    tool_name = match.get("tool_name", match.get("name", "unknown"))
    vertical   = match.get("vertical", "")
    desc       = match.get("description", "")[:80]
    print(f"  [{vertical}] {tool_name}")
    if desc:
        print(f"    {desc}")

## Create an Agent Wallet

Agent wallets are autonomous payment accounts that an AI agent can control directly. The wallet can hold balances and initiate payments without human-in-the-loop approval for each transaction — useful for automated business workflows.

In [ ]:
wallet_result = await call_mcp_tool(
    "wallet_create",
    {
        "agent_id": "cookbook-demo-agent",
        "wallet_name": "Business Ops Wallet",
        "currency": "USD",
        "spending_limit_usd": 1000.00
    }
)

print("Wallet created:")
print(json.dumps(wallet_result, indent=2))

## Process a Payment

`pay_universal` is a single tool that handles payments to any entity across multiple rails (ACH, wire, crypto, card). The agent specifies intent; HiveAgent handles protocol selection and execution.

In [ ]:
payment_result = await call_mcp_tool(
    "pay_universal",
    {
        "recipient": "Acme Supplies Inc",
        "amount_usd": 250.00,
        "memo": "Invoice #INV-2026-0042 — Q1 office supplies",
        "preferred_rail": "ach"
    }
)

print("Payment result:")
print(json.dumps(payment_result, indent=2))

## Smart Payment Routing

`route_payment` compares payment rails (ACH, wire, USDC, card) and recommends the optimal one based on speed, cost, and recipient capabilities. This is useful when the agent needs to balance cost vs. settlement time.

In [ ]:
routing_result = await call_mcp_tool(
    "route_payment",
    {
        "amount_usd": 15000.00,
        "recipient_country": "DE",
        "urgency": "same_day",
        "optimize_for": "cost"   # or 'speed', 'reliability'
    }
)

print("Routing recommendation:")
recommended = routing_result.get("recommended_rail", {})
print(f"  Protocol : {recommended.get('rail', routing_result.get('rail', 'N/A'))}")
print(f"  Est. fee : ${recommended.get('estimated_fee_usd', routing_result.get('estimated_fee_usd', 'N/A'))}")
print(f"  Settlement: {recommended.get('settlement_time', routing_result.get('settlement_time', 'N/A'))}")

print("\nFull response:")
print(json.dumps(routing_result, indent=2))

## Using Claude as the Orchestrator

Now we wire everything together: Claude decides which MCP tools to call and in what order. We expose the discovered tools as Claude tool definitions and let the model drive the workflow.

In [ ]:
import os

# Build tool definitions from the MCP catalog
# We'll use a small curated subset for this demo
claude_tools = [
    {
        "name": "hiveagent_discover",
        "description": "Search for relevant HiveAgent tools using a natural language query.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Natural language description of the task"}
            },
            "required": ["query"]
        }
    },
    {
        "name": "wallet_create",
        "description": "Create an autonomous payment wallet for an AI agent.",
        "input_schema": {
            "type": "object",
            "properties": {
                "agent_id":          {"type": "string"},
                "wallet_name":       {"type": "string"},
                "currency":          {"type": "string", "default": "USD"},
                "spending_limit_usd": {"type": "number"}
            },
            "required": ["agent_id", "wallet_name"]
        }
    },
    {
        "name": "route_payment",
        "description": "Compare payment rails and recommend the optimal protocol for a transaction.",
        "input_schema": {
            "type": "object",
            "properties": {
                "amount_usd":        {"type": "number"},
                "recipient_country": {"type": "string"},
                "urgency":           {"type": "string", "enum": ["normal", "same_day", "instant"]},
                "optimize_for":      {"type": "string", "enum": ["cost", "speed", "reliability"]}
            },
            "required": ["amount_usd"]
        }
    },
    {
        "name": "pay_universal",
        "description": "Send a payment to any recipient across multiple payment rails.",
        "input_schema": {
            "type": "object",
            "properties": {
                "recipient":      {"type": "string"},
                "amount_usd":     {"type": "number"},
                "memo":           {"type": "string"},
                "preferred_rail": {"type": "string"}
            },
            "required": ["recipient", "amount_usd"]
        }
    }
]


async def run_agent(user_request: str) -> str:
    """Run a simple agentic loop: Claude calls MCP tools until the task is done."""
    client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
    messages = [{"role": "user", "content": user_request}]

    print(f"User: {user_request}\n")

    while True:
        response = client.messages.create(
            model="claude-opus-4-5",
            max_tokens=1024,
            tools=claude_tools,
            messages=messages
        )

        if response.stop_reason == "end_turn":
            final_text = "".join(
                block.text for block in response.content if hasattr(block, "text")
            )
            print(f"Agent: {final_text}")
            return final_text

        # Process tool calls
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                print(f"  -> calling {block.name}({json.dumps(block.input, separators=(',', ':'))})")
                try:
                    result = await call_mcp_tool(block.name, block.input)
                except Exception as e:
                    result = {"error": str(e)}
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps(result)
                })

        # Append assistant turn + tool results and loop
        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": tool_results})


print("Agent loop defined.")

In [ ]:
# Run the agent on a business operations task
await run_agent(
    "I need to pay our German supplier €12,000 by end of day. "
    "Find the cheapest same-day payment route and create a wallet for our procurement agent."
)

## Summary

This notebook demonstrated how to:

1. **Call MCP tools with raw `httpx`** — no third-party MCP SDK needed; just JSON-RPC 2.0 POST requests
2. **Discover tools dynamically** — use `hiveagent_discover` to find relevant tools by natural language
3. **Create agent wallets** — autonomous payment accounts that Claude can manage
4. **Process payments** — send to any recipient via any rail with `pay_universal`
5. **Route intelligently** — compare cost, speed, and reliability across protocols
6. **Orchestrate with Claude** — use the Anthropic SDK's tool-use loop to let Claude drive the workflow

**Key dependencies:** only `anthropic` and `httpx` — standard packages that follow the repo's convention.

**Learn more:**
- [HiveAgent documentation](https://hiveagentiq.com/docs)
- [Try the live playground](https://hiveagentiq.com/playground.html)
- [MCP specification](https://modelcontextprotocol.io)